# Replicate Figure 5 of Barumerli et al. (2023) — paper Fig. 4

This notebook reproduces Figure 5 of the original model paper (https://doi.org/10.1051/aacus/2023006)
with the Python implementation, for the five ARI listeners of Majdak et al. (2010), and compares the
model predictions with the measured data and with the original MATLAB (AMT) implementation across
spatial sectors.

* **Output:** `results/replicate_fig5_estimations.pkl` (model estimations; Git LFS) and the
  per-sector mean ± SE table reported in Sec. 3.1 (e.g. polar error at 90°: 40.2 / 43.5 / 38.5 / 43.2°).
* **Figure:** Fig. 4 is drawn by `figures/fig_replicate5.py` from the same cache.
* **Inputs:** `data/reference/majdak2010.mat` (measured data, AMT auxdata), `results/barumerli2023_fig5.mat`
  (MATLAB model output) and — only if the cache is missing — the ARI SOFA files in `data/ari/`
  (all downloaded by `init.sh`).

The shipped cache contains the simulations used for the published figure. They were computed with a
development version of `bayesian_listener` preceding 0.2.0; regenerating the cache with 0.2.0 gives
statistically equivalent but not identical numbers (RNG and motor-noise sampling changed in 0.2.0).

In [1]:
import pickle
import time

import numpy as np
import pandas as pd
import pyfar as pf
from scipy.io import loadmat
from scipy import stats as sp_stats

from bayesian_listener import BayesianListener, metrics, utils

import paths

print("✓ Imports complete")

✓ Imports complete


In [2]:
# Model parameters for each subject and variant
# Amplitude variant (original paper)
sigma_amp = {
    'nh12': {
        'sigma_itd': 0.569,
        'sigma_ild': 0.5,
        'sigma_spectral': 3.4,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(8.5) ** 2)
    },
    'nh15': {
        'sigma_itd': 0.569,
        'sigma_ild': 0.5,
        'sigma_spectral': 3.2,
        'sigma_prior': 10.0,
        'kappa_motor': 1.0 / (np.deg2rad(14.27) ** 2)
    },
    'nh16': {
        'sigma_itd': 0.569,
        'sigma_ild': 1.0,
        'sigma_spectral': 3.6,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(11.0) ** 2)
    },
    'nh17': {
        'sigma_itd': 0.569,
        'sigma_ild': 0.5,
        'sigma_spectral': 4.1,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(14.3) ** 2)
    },
    'nh18': {
        'sigma_itd': 0.569,
        'sigma_ild': 1.0,
        'sigma_spectral': 6.5,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(14.0) ** 2)
    }
}

# Gradient variant (spectral gradient parameters)
sigma_grad = {
    'nh12': {
        'sigma_itd': 0.569,
        'sigma_ild': 0.5,
        'sigma_spectral': 1.1,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(8.5) ** 2)
    },
    'nh15': {
        'sigma_itd': 0.569,
        'sigma_ild': 0.5,
        'sigma_spectral': 1.25,
        'sigma_prior': 11.0,
        'kappa_motor': 1.0 / (np.deg2rad(14.3) ** 2)
    },
    'nh16': {
        'sigma_itd': 0.569,
        'sigma_ild': 1.0,
        'sigma_spectral': 1.25,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(11.5) ** 2)
    },
    'nh17': {
        'sigma_itd': 0.569,
        'sigma_ild': 1.0,
        'sigma_spectral': 1.6,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(14.0) ** 2)
    },
    'nh18': {
        'sigma_itd': 0.569,
        'sigma_ild': 1.0,
        'sigma_spectral': 2.1,
        'sigma_prior': 11.5,
        'kappa_motor': 1.0 / (np.deg2rad(15.0) ** 2)
    }
}

sofa_file = {key: str(paths.ari_sofa(key)) for key in sigma_amp}  # data/ari/ARI_NH??_hrtf_M_dtf 256.sofa


In [3]:
# Model estimations for both variants: load the shipped cache, or recompute it.
# The cache stores plain numpy arrays keyed by (variant, subject):
#   (target directions, cartesian (n_targets, 3); estimations, cartesian (n_targets * reps, 3))
reps = 300        # repetitions per target
BASE_SEED = 42
convention = {'amp': 'Barumerli2023', 'grad': 'barumerli2023pge'}
cache_file = paths.RESULTS_DIR / 'replicate_fig5_estimations.pkl'


def cut_template_grid(target_coords):
    """Default 64-design template grid without the unmeasured bottom cap
    (as in the original AMT model: no barumerli2023 extrapolation below the
    lowest measured elevation)."""
    dirs = utils.load_n_design(64)
    dirs = pf.Coordinates.from_cartesian(dirs[:, 0], dirs[:, 1], dirs[:, 2])
    cut = dirs.cartesian[dirs.elevation > np.min(target_coords.elevation), :]
    return pf.Coordinates.from_cartesian(cut[:, 0], cut[:, 1], cut[:, 2])


def simulate(variant, sbj, params):
    m = BayesianListener(sofa_file[sbj], **params)
    m.compute_target(convention=convention[variant], use_cache=False)
    m.compute_template(interpolation='barumerli2023',
                       interpolation_grid=cut_template_grid(m.target.coords),
                       use_cache=False)
    posterior = m.infer(repetitions=reps, seed=BASE_SEED)
    estimations = m.estimate(posterior, seed=BASE_SEED)
    return m.target.coords.cartesian, estimations.cartesian.reshape(-1, 3)


if cache_file.exists():
    print(f"Loading cached estimations from {cache_file.relative_to(paths.ROOT)}")
    with open(cache_file, 'rb') as f:   # only unpickle files from trusted sources
        raw = pickle.load(f)
else:
    print("Cache not found: simulating 5 subjects x 2 variants (needs data/ari/*.sofa)...")
    raw = {}
    for variant, sigma in [('amp', sigma_amp), ('grad', sigma_grad)]:
        for sbj in sofa_file:
            t0 = time.time()
            raw[(variant, sbj)] = simulate(variant, sbj, sigma[sbj])
            print(f"  {variant}/{sbj}: {time.time() - t0:.1f}s")
    with open(cache_file, 'wb') as f:
        pickle.dump(raw, f)
    print(f"✓ Saved {cache_file.relative_to(paths.ROOT)}")

# targets (repeated per repetition) and estimations as pyfar Coordinates
targets = {'amp': {}, 'grad': {}}
estims = {'amp': {}, 'grad': {}}
for (variant, sbj), (model_coords, estims_flat) in raw.items():
    t = np.repeat(model_coords, reps, axis=0)
    targets[variant][sbj] = pf.Coordinates.from_cartesian(t[:, 0], t[:, 1], t[:, 2])
    estims[variant][sbj] = pf.Coordinates.from_cartesian(
        estims_flat[:, 0], estims_flat[:, 1], estims_flat[:, 2])

print(f"✓ Estimations for {len(estims['amp'])} subjects × 2 variants")

Loading cached estimations from results/replicate_fig5_estimations.pkl
✓ Estimations for 5 subjects × 2 variants


In [4]:
# Compute model metrics by spatial sector for both variants
metrics_to_compute = ['rmsL', 'rmsPmedianlocal', 'querrMiddlebrooks']
#latitude_boundaries = np.deg2rad([-90, -40, 0, 40, 90])
#latitude_labels = [-65, -20, 20, 65]
latitude_boundaries = np.deg2rad([-90, -30, 30, 90])
latitude_labels = [-60, 0, 60]
polar_boundaries = np.deg2rad([-30, 30, 150, 210])
polar_labels = [0, 90, 180]

computed_metrics = {'amp': {}, 'grad': {}}

for variant in ['amp', 'grad']:
    for sbj in sofa_file:
        computed_metrics[variant][sbj] = {}

        for metric in metrics_to_compute:
            true_hp = targets[variant][sbj].spherical_side
            est_hp = estims[variant][sbj].spherical_side

            if metric == 'rmsL':
                bounds, labels = latitude_boundaries, latitude_labels
                angle_true = true_hp[:, 0]
            else:
                bounds, labels = polar_boundaries, polar_labels
                angle_true = true_hp[:, 1]

            computed_metrics[variant][sbj][metric] = {}
            for i, label in enumerate(labels):
                mask = (angle_true >= bounds[i]) & (angle_true < bounds[i + 1])
                if not np.any(mask):
                    computed_metrics[variant][sbj][metric][label] = {
                        'value': np.nan,
                        'metadata': {'output_unit': 'radians'}
                    }
                    continue

                metric_func = metrics.METRIC_FUNCTIONS[metric]
                value, aux_out = metric_func(true_hp[mask], est_hp[mask])
                computed_metrics[variant][sbj][metric][label] = {
                    'value': np.rad2deg(value) if metric_func._metadata['output_unit'] == 'radians' else value,
                    'metadata': metric_func._metadata
                }

print("✓ Computed model metrics for both variants")

✓ Computed model metrics for both variants


In [5]:
# Load measured behavioral data from Majdak et al. (2010) (AMT auxdata)
mat_data = loadmat(paths.REFERENCE_DIR / 'majdak2010.mat', struct_as_record=False, squeeze_me=True)
subjects_mat = mat_data["subject"]
condition_index = 4  # Free-field condition

measured_metrics = {}
for subj_struct in subjects_mat:
    id_mat = subj_struct.id.lower()
    if id_mat not in computed_metrics['amp'].keys():
        continue

    exp = getattr(subj_struct, 'expData')[condition_index].real
    if exp.ndim == 1:
        exp = exp.reshape(-1, 9)

    # columns 0:2 / 2:4 are target / response (azimuth, elevation) in degrees
    _true_arr = np.hstack([np.deg2rad(exp[:, 0:2]), np.ones((exp.shape[0], 1))])
    true = pf.Coordinates.from_spherical_elevation(_true_arr[:, 0], _true_arr[:, 1], _true_arr[:, 2])
    _est_arr = np.hstack([np.deg2rad(exp[:, 2:4]), np.ones((exp.shape[0], 1))])
    est = pf.Coordinates.from_spherical_elevation(_est_arr[:, 0], _est_arr[:, 1], _est_arr[:, 2])
    true_hp, est_hp = true.spherical_side, est.spherical_side

    measured_metrics[id_mat] = {}
    for metric in metrics_to_compute:
        if metric == 'rmsL':
            bounds, labels = latitude_boundaries, latitude_labels
            angle_true = true_hp[:, 0]
        else:
            bounds, labels = polar_boundaries, polar_labels
            angle_true = true_hp[:, 1]

        measured_metrics[id_mat][metric] = {}
        for i, label in enumerate(labels):
            mask = (angle_true >= bounds[i]) & (angle_true < bounds[i + 1])
            if not np.any(mask):
                continue
            metric_func = metrics.METRIC_FUNCTIONS[metric]
            value, _ = metric_func(true_hp[mask], est_hp[mask])
            measured_metrics[id_mat][metric][label] = {
                'value': np.rad2deg(value) if metric_func._metadata['output_unit'] == 'radians' else value
            }

print(f"✓ Loaded measured data for {len(measured_metrics)} subjects")

✓ Loaded measured data for 5 subjects


In [6]:
# Load MATLAB model results from barumerli2023_fig5.mat
mat_file = loadmat(paths.RESULTS_DIR / 'barumerli2023_fig5.mat', struct_as_record=False, squeeze_me=True)
fig5 = mat_file['cache'].value  # 5×2 array of Nx8 matrices (degrees)

# Subject order matches MATLAB (data_majdak2010 subjects 6–10)
subjects_order = list(computed_metrics['amp'].keys())  # ['nh12', 'nh15', 'nh16', 'nh17', 'nh18']
variants_map = {0: 'amp', 1: 'grad'}  # col 0 = DTF/amp, col 1 = PGE/grad

matlab_metrics = {variant: {} for variant in ['amp', 'grad']}

for s_idx, sbj in enumerate(subjects_order):
    for v_idx, variant in variants_map.items():
        data_mat = fig5[s_idx, v_idx]  # Nx8 matrix in degrees

        # Columns (0-indexed): 4=targ_lat, 5=targ_pol, 6=resp_lat, 7=resp_pol
        true_hp = np.column_stack([np.deg2rad(data_mat[:, 4]),
                                   np.deg2rad(data_mat[:, 5]),
                                   np.ones(len(data_mat))])
        est_hp = np.column_stack([np.deg2rad(data_mat[:, 6]),
                                  np.deg2rad(data_mat[:, 7]),
                                  np.ones(len(data_mat))])

        matlab_metrics[variant][sbj] = {}
        for metric in metrics_to_compute:
            if metric == 'rmsL':
                bounds, labels = latitude_boundaries, latitude_labels
                angle_true = true_hp[:, 0]
            else:
                bounds, labels = polar_boundaries, polar_labels
                angle_true = true_hp[:, 1]

            matlab_metrics[variant][sbj][metric] = {}
            for i, label in enumerate(labels):
                mask = (angle_true >= bounds[i]) & (angle_true < bounds[i + 1])
                if not np.any(mask):
                    matlab_metrics[variant][sbj][metric][label] = {'value': np.nan}
                    continue

                metric_func = metrics.METRIC_FUNCTIONS[metric]
                value, _ = metric_func(true_hp[mask], est_hp[mask])
                matlab_metrics[variant][sbj][metric][label] = {
                    'value': np.rad2deg(value) if metric_func._metadata['output_unit'] == 'radians' else value
                }

print(f"✓ Loaded MATLAB metrics for {len(subjects_order)} subjects × 2 variants")

✓ Loaded MATLAB metrics for 5 subjects × 2 variants


## Aggregated results (Fig. 4)

Mean ± SE across the five listeners per sector; these are the numbers quoted in Sec. 3.1.
The figure itself is produced by `figures/fig_replicate5.py`.

In [7]:
subjects_order = list(sofa_file)
metric_titles = ['Lateral error (deg)', 'Polar error (deg)', 'Quadrant error (%)']
sector_positions = {
    'rmsL': latitude_labels,
    'rmsPmedianlocal': polar_labels,
    'querrMiddlebrooks': polar_labels,
}


def collect_values(source_dict, variant, metric, x_vals):
    """Gather per-subject values into an array (n_subjects × n_sectors)."""
    return np.array([
        [source_dict[variant][sbj][metric].get(x, {}).get('value', np.nan) for x in x_vals]
        for sbj in subjects_order
    ])


table_data = []
for row, metric in enumerate(metrics_to_compute):
    x_vals = sector_positions[metric]
    data_map = {
        'Measured': np.array([[measured_metrics[sbj][metric].get(x, {}).get('value', np.nan)
                               for x in x_vals] for sbj in subjects_order]),
        'Python amp': collect_values(computed_metrics, 'amp', metric, x_vals),
        'MATLAB amp': collect_values(matlab_metrics, 'amp', metric, x_vals),
        'Python grad': collect_values(computed_metrics, 'grad', metric, x_vals),
        'MATLAB grad': collect_values(matlab_metrics, 'grad', metric, x_vals),
    }
    for i, x in enumerate(x_vals):
        row_entry = {'Metric': metric_titles[row], 'Angle (deg)': x}
        for label, data in data_map.items():
            row_entry[label] = f"{np.nanmean(data[:, i]):.2f} ± {sp_stats.sem(data[:, i], nan_policy='omit'):.2f}"
        table_data.append(row_entry)

df_results = pd.DataFrame(table_data).set_index(['Metric', 'Angle (deg)'])
print("Aggregated results: mean ± SE across subjects")
print(df_results.to_string())

Aggregated results: mean ± SE across subjects
                                     Measured    Python amp    MATLAB amp   Python grad   MATLAB grad
Metric              Angle (deg)                                                                      
Lateral error (deg) -60          12.96 ± 0.92  13.33 ± 1.10  13.35 ± 1.38  13.46 ± 1.19  13.61 ± 1.57
                     0           11.73 ± 1.21  12.82 ± 1.08  12.54 ± 1.12  13.05 ± 1.13  12.84 ± 1.22
                     60          13.30 ± 0.98  13.22 ± 1.05  13.16 ± 1.03  13.35 ± 1.16  12.65 ± 1.00
Polar error (deg)    0           21.26 ± 2.71  15.66 ± 1.51  15.35 ± 1.70  14.52 ± 1.54  14.33 ± 1.53
                     90          42.59 ± 1.82  40.19 ± 2.62  43.50 ± 2.97  38.53 ± 2.73  43.24 ± 2.26
                     180         18.68 ± 2.64  16.96 ± 1.20  16.81 ± 1.56  15.30 ± 1.50  14.35 ± 1.31
Quadrant error (%)   0            6.28 ± 2.53   1.82 ± 1.27   0.91 ± 0.81   1.60 ± 0.91   1.27 ± 1.01
                     90           8.